## Manual Prompting Analysis

This is a "manual prompting" analysis notebook analyzing the differences between the ground-truth dataset and the cleaned (by manual prompting) dataset. 

MP dataset used Google Gemini's web/chat interface (Gemini 3, Thinking) at gemini.google.com. Refer to `/cleaning/fdic_cleaning/manual_prompting.ipynb` for more details.

#### Analysis

In [86]:
# import packages
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [87]:
# import data (only 250 rows for now)
df_manual = pd.read_csv("../../data/fdic/mt_cleaned_fdic.csv")
df_llm = pd.read_csv("../../data/fdic/mp_cleaned_fdic.csv")

In [88]:
# schema integrity
manual_cols = set(df_manual.columns)
llm_cols = set(df_llm.columns)

schema_diff = {
    "dropped_columns": list(manual_cols - llm_cols),
    "invented_columns": list(llm_cols - manual_cols),
    "row_count_diff": len(df_llm) - len(df_manual)
}

print("schema differences:")
print("-"*20)
print(schema_diff)

schema differences:
--------------------
{'dropped_columns': ['change_code_3', 'change_code_4', 'change_code_2', 'end_effective_date'], 'invented_columns': [], 'row_count_diff': -1}


In [89]:
# make sure both csvs have same "primary key" due to comparisions
df_manual = df_manual.set_index("fdic_certificate_number")
df_llm = df_llm.set_index("fdic_certificate_number")

commons = df_manual.index.intersection(df_llm.index)

df_manual = df_manual.loc[commons]
df_llm = df_llm.loc[commons]

In [90]:
# calculate metrics for all cols
precisions = []
recalls = []
f1s = []

for col in df_llm.columns:
    
    if col not in df_manual.columns:
        continue
    
    y_true = df_manual[col].fillna("MISSING").astype(str)
    y_pred = df_llm[col].fillna("MISSING").astype(str)
    
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)

In [91]:
# find avgs for all cols
avg_precision = np.mean(precisions)
avg_recall = np.mean(recalls)
avg_f1 = np.mean(f1s)

print("\navg metrics for all cols:")
print("-"*25)
print(f"avg precision: {avg_precision:.4f}")
print(f"avg recall:    {avg_recall:.4f}")
print(f"avg F1 score:  {avg_f1:.4f}")


avg metrics for all cols:
-------------------------
avg precision: 0.9693
avg recall:    0.9671
avg F1 score:  0.9671


In [92]:
# misinterpretation rate: the proportion of records with 1+ violation/incorrect repairs across all columns

common_cols = [col for col in df_llm.columns if col in df_manual.columns]

# a row is "misinterpreted" if ANY column value differs from ground truth
mismatches = (df_manual[common_cols].fillna("MISSING").astype(str) !=
              df_llm[common_cols].fillna("MISSING").astype(str))

misinterpreted_rows = mismatches.any(axis=1).sum()
total_rows = len(df_manual)
misinterpretation_rate = misinterpreted_rows / total_rows

print("misinterpretations:")
print("-"*20)
print(f"rows: {misinterpreted_rows} / {total_rows}")
print(f"rate: {misinterpretation_rate:.4f} ({misinterpretation_rate*100:.2f}%)")

misinterpretations:
--------------------
rows: 248 / 249
rate: 0.9960 (99.60%)


In [93]:
# column-level mismatches
col_mismatch_rates = mismatches.mean().sort_values(ascending=False)

In [94]:
print("top 15 most mismatched columns:")
print("-"*35)
print(col_mismatch_rates.head(15).to_string())

top 15 most mismatched columns:
-----------------------------------
cbsa_division_fips_code    0.771084
change_code_1              0.751004
csa_fips_code              0.369478
institution_name           0.341365
cbsa_micro_flag            0.216867
cbsa_fips_code             0.216867
csa_indicator              0.216867
rssd_id                    0.180723
offices_count              0.104418
net_income                 0.012048
row_pretax_quarterly       0.012048
return_on_equity           0.012048
roe_quarterly              0.012048
parent_parcert             0.012048
total_domestic_deposits    0.012048


In [95]:
print("bottom 15 least mismatched columns (most accurate):")
print("-"*51)
print(col_mismatch_rates.tail(15).to_string())

bottom 15 least mismatched columns (most accurate):
---------------------------------------------------
trade_name_1                 0.0
state_fips_code              0.0
state                        0.0
law_sasser                   0.0
sasser_institute             0.0
run_date                     0.0
denovo_institute             0.0
fdic_geo_region              0.0
state_name                   0.0
fdic_supervisory_region      0.0
state_chartered              0.0
reporting_period_end_date    0.0
report_date                  0.0
regulator                    0.0
cbsa_division_flag           0.0


In [96]:
print("none or total mismatches:")
print("-"*25)
print(f"columns with 0% mismatch: {(col_mismatch_rates == 0).sum()}")
print(f"columns with 100% mismatch: {(col_mismatch_rates == 1).sum()}")

none or total mismatches:
-------------------------
columns with 0% mismatch: 76
columns with 100% mismatch: 0


In [97]:
# summarize mismatches by columns
print(f"total shared columns: {len(common_cols)}")
print(f"columns with 0% mismatch: {(col_mismatch_rates == 0).sum()} ({(col_mismatch_rates == 0).mean()*100:.1f}%)")
print(f"columns with <5% mismatch: {(col_mismatch_rates < 0.05).sum()} ({(col_mismatch_rates < 0.05).mean()*100:.1f}%)")
print(f"columns with 5-25% mismatch: {((col_mismatch_rates >= 0.05) & (col_mismatch_rates < 0.25)).sum()}")
print(f"columns with >25% mismatch: {(col_mismatch_rates >= 0.25).sum()}")

total shared columns: 103
columns with 0% mismatch: 76 (73.8%)
columns with <5% mismatch: 94 (91.3%)
columns with 5-25% mismatch: 5
columns with >25% mismatch: 4
